In [ ]:
import pandas as pd 
from scipy.spatial import distance
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time
from multiprocessing import Pool
import tqdm
from itertools import combinations
from scipy import stats
import seaborn as sns; sns.set_theme(style='white')
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.colors import LogNorm, Normalize
from matplotlib.ticker import MaxNLocator
from Levenshtein import distance as lev
from matplotlib_venn import venn2
import networkx as nx
AF2= pd.read_pickle('../files/AF2_with_ENSG.pkl')
#df = pd.read_csv('GPCR_ThT/RNAseq/BABU-301366-STRANDED_RSEM_gene_TPM.2023-07-18_23-16-12.txt',delimiter='\t')

#GSEA

import gseapy as gp
from gseapy.plot import gseaplot
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [ ]:
df = pd.read_csv('/Users/aelazar/Downloads/BABU-301366-STRANDED_RSEM_gene_count.2023-07-18_23-16-12.txt',delimiter='\t')
#df = df[df.bioType=='protein_coding']
d = {}
for i in df.loc[:,['geneID','geneSymbol']].set_index('geneID').index:
    d[i] = df[df.geneID==i].geneSymbol.values[0]

In [ ]:
def fdr(p_vals):

    from scipy.stats import rankdata
    ranked_p_values = rankdata(p_vals)
    fdr = p_vals * len(p_vals) / ranked_p_values
    fdr[fdr > 1] = 1

    return fdr

In [ ]:
counts = df.set_index('geneID')
counts= counts.loc[:,['2613875_5_HTR7_WT_basal_rep_1', '2613876_5_HTR7_WT_basal_rep_2',
       '2613877_5_HTR7_WT_basal_rep_3', '2613878_5_HTR7_MUT_basal_rep_1',
       '2613879_5_HTR7_MUT_basal_rep_2', '2613880_5_HTR7_MUT_basal_rep_3','2613881_HEK_pCDNA_rep_1','2613882_HEK_pCDNA_rep_2','2613883_HEK_pCDNA_rep_3']]#,'2613881_HEK_pCDNA_rep_1','2613881_HEK_pCDNA_rep_2',
#                     '2613883_HEK_pCDNA_rep_3']]
counts = counts.astype(np.int64)
counts = counts[counts.sum(axis = 1) > 10]


print(len(counts))
counts = counts.T

metadata = pd.DataFrame(zip(counts.index, ['WT','WT','WT', 'MUT', 'MUT', 'MUT','C','C','C']),
                        columns = ['Sample','Condition'])
metadata = metadata.set_index('Sample')
metadata

In [ ]:
counts.to_csv('GPCR_ThT/RNAseq/RNA_seq_counts.csv')

In [ ]:
counts

In [ ]:
dds = DeseqDataSet(counts=counts,
            clinical=metadata,
            design_factors="Condition")
dds.deseq2()

In [ ]:
stat_res = DeseqStats(dds, n_cpus=8, contrast = ('Condition','MUT','WT'))
stat_res.summary()
res = stat_res.results_df
res['Symbol'] = res.index.map(d) 
res= res[res.baseMean >= 10]
#res['padj'] = fdr(res['pvalue'])
ranking = res[['Symbol', 'stat']].dropna().sort_values('stat', ascending = False)
ranking = ranking.drop_duplicates('Symbol')
ranking = ranking.drop_duplicates('stat')

In [ ]:
dds

In [ ]:

fig, ax = plt.subplots(figsize=(2.5,2.5)) 
#plt.suptitle('A')
sc.pl.pca(dds, color = 'Condition', size = 200,ax=ax,annotate_var_explained=True,save='.svg')


In [ ]:
res.sort_values(by='padj')#.to_csv('deseq2_anaysis.csv')

In [ ]:
pre_res = gp.prerank(rnk = ranking, gene_sets = 'KEGG_2021_Human', seed = 4, permutation_num = 1000)
out_df = pre_res.res2d
out_df['abs_NES'] = np.abs(out_df['NES'])
#out_df = out_df[out_df['FDR q-val']<.15].sort_values('abs_NES',ascending = False).reset_index(drop = True)


In [ ]:
out_df[out_df.Term=='MAPK signaling pathway']

In [ ]:
out_df[(out_df['FDR q-val']<.2)&(out_df.abs_NES>1.5)]

In [ ]:
pre_res = gp.prerank(rnk = ranking, gene_sets = 'KEGG_2021_Human', seed = 4, permutation_num = 1000)


In [ ]:
out_df['gene_pre'] = [float(i.split('%')[0]) for i in out_df['Gene %']]
#out_df[out_df.NES<-1.70].sort_values('gene_pre',ascending = False)

In [ ]:
out_df#[out_df.Term=='MAPK signaling pathway']

In [ ]:
out_df[out_df['FDR q-val']<.15].to_csv('GPCR_ThT/RNAseq/GESA_significant.csv')

In [ ]:
def flatten(l):
    return [item for sublist in l for item in sublist]

In [ ]:
z = []
x =pd.Series(flatten([i.split(';') for i in out_df[out_df.NES<-1.7].Lead_genes.to_list()])).value_counts()
for i in x.index:
    z.append({'gene':i,'No':x.loc[i]})
zz = pd.DataFrame(z)
zz

In [ ]:
def jaccard_similarity(list1, list2):
    s1 = set(list1)
    s2 = set(list2)
    #print(len(s1.intersection(s2)))
    return float(len(s1.intersection(s2)) / len(s1.union(s2)))

In [ ]:
l = []
for i in out_df[out_df.NES<-1.7].index:
    for j in out_df[out_df.NES<-1.7].index:
        
        list1 = out_df.loc[i].Lead_genes.split(';')
        list2 = out_df.loc[j].Lead_genes.split(';')
        if i==j:
            l.append({'pathway1':out_df.loc[i,'Term'],'pathway2':out_df.loc[j,'Term'],'jaccard_similarity':
                 1})
        else:
            l.append({'pathway1':out_df.loc[i,'Term'],'pathway2':out_df.loc[j,'Term'],'jaccard_similarity':
                 jaccard_similarity(list1, list2)})

In [ ]:
import matplotlib.colors as colors
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#18827E",'gray',"#B34F91"])#"#"

In [ ]:
l = []
for i in out_df[out_df.NES<-1.7].index:
    for j in out_df[out_df.NES<-1.7].index:
        
        list1 = out_df.loc[i].Lead_genes.split(';')
        list2 = out_df.loc[j].Lead_genes.split(';')
        if i==j:
            l.append({'pathway1':out_df.loc[i,'Term'],'pathway2':out_df.loc[j,'Term'],'jaccard_similarity':
                 1})
        else:
            l.append({'pathway1':out_df.loc[i,'Term'],'pathway2':out_df.loc[j,'Term'],'jaccard_similarity':
                 jaccard_similarity(list1, list2)})
z = pd.DataFrame(l).pivot_table(values='jaccard_similarity',
    index='pathway1',
    columns='pathway2')
#mask = np.triu(np.ones_like(z, dtype=bool))
sns.set(font_scale=.85,style="white")
g = sns.clustermap(z,cmap=ww,dendrogram_ratio=.1,figsize=(6, 6),lw=0,norm=colors.SymLogNorm(linthresh=.15, linscale=1,
                                              vmin=0, vmax=1, base=10))
lab_y = [i.get_text().replace(' ','\,').replace('-','‑') for i in g.ax_heatmap.get_yticklabels()]
g.ax_heatmap.set_yticklabels(['$%s$'%i for i in lab_y],fontsize=10)
g.ax_heatmap.set_xticklabels(['$%s$'%i for i in lab_y],fontsize=10)
# mask = np.triu(np.ones_like(z))#
# values = g.ax_heatmap.collections[0].get_array().reshape(z.shape)
# new_values = np.ma.array(values, mask=mask)
# g.ax_heatmap.collections[0].set_array(new_values)
#g.ax_row_dendrogram.set_visible(False)
plt.savefig('figures/heat_dowm.svg')

In [ ]:
l = []
for i in out_df[out_df.NES>1.7].index:
    for j in out_df[out_df.NES>1.7].index:
        
        list1 = out_df.loc[i].Lead_genes.split(';')
        list2 = out_df.loc[j].Lead_genes.split(';')
        if i==j:
            l.append({'pathway1':out_df.loc[i,'Term'],'pathway2':out_df.loc[j,'Term'],'jaccard_similarity':
                 1})
        else:
            l.append({'pathway1':out_df.loc[i,'Term'],'pathway2':out_df.loc[j,'Term'],'jaccard_similarity':
                 jaccard_similarity(list1, list2)})
z = pd.DataFrame(l).pivot_table(values='jaccard_similarity',
    index='pathway1',
    columns='pathway2')
#mask = np.triu(np.ones_like(z, dtype=bool))
sns.set(font_scale=.85,style="white")
g = sns.clustermap(z,cmap=ww,dendrogram_ratio=.1,figsize=(6, 6),lw=0,norm=colors.SymLogNorm(linthresh=.15, linscale=1,
                                              vmin=0, vmax=1, base=10),cbar_kws={'label': 'JSC',"shrink": .32})
#lab_x = [int(str(i.get_text()).split()[0]) for i in g.ax_heatmap.get_yticklabels()]
lab_y = [i.get_text().replace(' ','\,').replace('-','‑') for i in g.ax_heatmap.get_yticklabels()]
g.ax_heatmap.set_yticklabels(['$%s$'%i for i in lab_y],fontsize=10)
g.ax_heatmap.set_xticklabels(['$%s$'%i for i in lab_y],fontsize=10)
# mask = np.triu(np.ones_like(z))#
# values = g.ax_heatmap.collections[0].get_array().reshape(z.shape)
# new_values = np.ma.array(values, mask=mask)
# g.ax_heatmap.collections[0].set_array(new_values)
#g.ax_row_dendrogram.set_visible(False)
plt.savefig('figures/heat_up.svg')

In [ ]:
[i.get_text().replace(' ','\,').replace('-','‑') for i in g.ax_heatmap.get_yticklabels()]

In [ ]:
np.fill_diagonal(z.values, -np.inf)
#np.max(A, axis=1)
np.max(z, axis=1).max()

In [ ]:
out_df[out_df.Term.str.find('ndo')!=-1]
#out_df[out_df['FDR q-val']<.15].sort_values('Gene %',ascending = False)

In [ ]:
out_df

In [ ]:
# # out = []

# # for term in list(pre_res.results):
# #     out.append([term,
# #                pre_res.results[term]['fdr'],
# #                pre_res.results[term]['es'],
# #                pre_res.results[term]['nes']])

# # out_df = pd.DataFrame(out, columns = ['Term','fdr', 'es', 'nes']).sort_values('nes',ascending = False).reset_index(drop = True)
# out_df.loc[15]

In [ ]:
out_df.pivot_table(index='Term')#[out_df.abs_NES>1.7].iloc[[12,8]]

In [ ]:
#path_ = '/Users/aelazar/notebooks_for_paper/GPCR_ThT/RNAseq/file.png'
term_to_graph = out_df.iloc[[0]].Term
term_to_graph
axs = pre_res.plot(terms=term_to_graph,colors=['#194D63','#E13700','green','pink'],
legend_kws={'fontsize': 8}, # set the legend loc
show_ranking=True,figsize=(10,10))


In [ ]:
from gseapy import gseaplot2


In [ ]:
pre_res.results['Retrograde endocannabinoid signaling']['matched_genes']

In [ ]:
pd.Series(pre_res.results['Retrograde endocannabinoid signaling']['lead_genes'].split(';')).to_clipboard(index=0)

In [ ]:
t = []
for i in enumerate(pre_res.results['MAPK signaling pathway']['RES']):
    t.append({'c':i[0],'val':i[1],'pathway':'$MAPK\;signaling$'})
for i in enumerate(pre_res.results['Pathways of neurodegeneration']['RES']):
    t.append({'c':i[0],'val':i[1],'pathway':'$Pathways\;of\;neurodegeneration$'})

tt = []
for i in enumerate(ranking.Symbol):
    if i[0] in pre_res.results['MAPK signaling pathway']['hits']:
        #print('hit')
        tt.append({'c':i[0],'val':1,})
    else:
        tt.append({'c':i[0],'val':0})
t3 = []
for i in enumerate(ranking.Symbol):
    if i[0] in pre_res.results['Pathways of neurodegeneration']['hits']:
        #print('hit')
        t3.append({'c':i[0],'val':1,})
    else:
        t3.append({'c':i[0],'val':0})

In [ ]:
out_df[out_df.abs_NES>1.6]#.iloc[[12,8]]

In [ ]:
ww = matplotlib.colors.LinearSegmentedColormap.from_list("", ["#18827E",'gray',"#B34F91"])#"green purpule 

In [ ]:
import matplotlib.gridspec as gridspec
fig = plt.subplots(figsize=(6,4))
gs = gridspec.GridSpec(12, 12,figure=fig)
gs.update(left=0.05, right=0.48, wspace=0.0,hspace=0)
ax1 = plt.subplot(gs[0:9, :])
ax2 = plt.subplot(gs[10,:])
ax3 = plt.subplot(gs[11,:])
sns.lineplot(x='c',y='val',data=pd.DataFrame(t),hue='pathway',ax=ax1,palette=['#18827E','#B34F91'],lw=5)
ax_twin = ax1.twinx()
ax_twin.fill_between([i[0] for i in enumerate(res.stat.sort_values(ascending=False))],
                     y1=[i[1] for i in enumerate(res.stat.sort_values(ascending=False))], 
                 y2=0, color="#C9D3DB",alpha=.8)
sns.lineplot(x='c',y='val',data=pd.DataFrame(tt),lw=.15,ax=ax3,c='#18827E')
sns.lineplot(x='c',y='val',data=pd.DataFrame(t3),lw=.15,ax=ax2,c='#B34F91')
ax1.set_ylim([-.65,.65])
ax2.set_ylim([.3,.5])
ax3.set_ylim([.3,.5])
ax3.set_xlim([-100,7731+100])
ax2.set_xlim([-100,7731+100])
ax1.set_xlim([-100,7731+100])
ax1.set_xticklabels([])
ax1.text(5700,.5,'$NES=%.1f$'%out_df.loc[18,'NES'],fontsize=8,c='#B34F91')
ax1.text(5700,.4,'$NES=%.1f$'%out_df.loc[12,'NES'],fontsize=8,c='#194D63')
ax1.set_xlabel('')
ax2.set_xticks([])
ax2.set_yticks([])
#ax1.set_xticks([])
ax3.set_yticks([])
ax2.set_ylabel('')
ax3.set_ylabel('')
ax2.set_xlabel('')
ax1.legend(title='$KEGG\;pathway:$',fontsize=6,title_fontsize=6,loc='lower left')
ax1.grid()
ax3.set_xlabel('$Gene\;Rank$',fontsize=10)
ax1.set_ylabel('$Enrichment\;score$',fontsize=10)
ax_twin.set_ylabel('$Ranked\;metric$',fontsize=10)
ax_twin.spines['right'].set_color('#495E6E')
ax_twin.yaxis.label.set_color('#495E6E')
ax_twin.tick_params(axis='y', colors='#495E6E')
plt.savefig('GESA_TEMP.svg',bbox_inches='tight',transparent=True)

In [ ]:
# from gseapy import dotplot
# pre_res
# # to save your figure, make sure that ``ofname`` is not None
# ax = dotplot(pre_res.res2d,
#              column="FDR q-val",
#              title='KEGG_2016',
#              cmap=plt.cm.viridis,
#              size=3, # adjust dot size
#              figsize=(4,5), cutoff=0.25, show_ring=False)

In [ ]:
pd.DataFrame(tt)['c']

In [ ]:
dds.layers['log1p'] = np.log1p(dds.layers['normed_counts'])

In [ ]:
from sanbomics.plots import volcano


In [ ]:
stat_res = DeseqStats(dds, n_cpus=8, contrast = ('Condition','WT','C'))
stat_res.summary()
res = stat_res.results_df
res['Symbol'] = res.index.map(d) 
res_WT= res[res.baseMean >= 10]
# stat_res = DeseqStats(dds, n_cpus=8, contrast = ('Condition','MUT','C'))
# stat_res.summary()
# res = stat_res.results_df
# res['Symbol'] = res.index.map(d) 
# res_MUT= res[res.baseMean >= 10]

# res_WT['log_fdr']=-1*np.log10(res['padj'])
# res_MUT['log_fdr']=-1*np.log10(res['padj'])

#sns.scatterplot(x='log2FoldChange',y='log_fdr',data=res[res.Symbol.isin(zz[zz.No>1].gene)],color='r',edgecolor=None)

In [ ]:
from adjustText import adjust_text

In [ ]:
len(pre_res.results['MAPK signaling pathway']['matched_genes'].split(';'))

In [ ]:
fig = plt.figure(figsize = ((2.5),2.5))#20,10
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
res_WT['log_fdr']=-1*np.log10(res_WT['padj'])
res_WT['log_fdr_v2'] = -1*np.log10(fdr(res_WT['pvalue']))
ax = sns.scatterplot(x='log2FoldChange',y='log_fdr_v2',data=res_WT,edgecolor=None,alpha=.5,color='#C03830')
plt.title('$5‑HTR7\,vs\,pCDNA$',fontsize=10)
ax.grid()
texts = []
for i in res_WT[(res_WT.log_fdr_v2>1.3)&
                (res_WT.Symbol.isin(pre_res.results['MAPK signaling pathway']['matched_genes'].split(';')))].index:
    texts.append(plt.text(res_WT.loc[i,'log2FoldChange'],res_WT.loc[i,'log_fdr_v2'],
            '$%s$'%res_WT.loc[i,'Symbol'],
                          fontsize=7))
    
texts = []
for i in res_WT[(res_WT.log_fdr_v2>1.3)&
                (res_WT.Symbol.isin(pre_res.results['Pathways of neurodegeneration']['matched_genes'].split(';')))].index:
    texts.append(plt.text(res_WT.loc[i,'log2FoldChange'],res_WT.loc[i,'log_fdr_v2'],
            '$%s$'%res_WT.loc[i,'Symbol'],
                          fontsize=7,c='b'))
    
adjust_text(texts, autoalign='xy',force_text=(-.2, .1),force_points=(-.1, 0.1),
            only_move={'points':'xy', 'text':'xy'},
            arrowprops=dict(arrowstyle="-", color='k', lw=0.5),zorder=0)#,expand_align=(-1,-1)
plt.ylabel('$‑log_{10}(FDR)$',fontsize=10)
plt.xlabel('$log_{2}(FC)$',fontsize=10)
#plt.savefig('GPCR_ThT/RNAseq/WT_vs_Control_vp.svg',bbox_inches='tight',transparent=True)

In [ ]:
! open GPCR_ThT/RNAseq

In [ ]:
stat_res = DeseqStats(dds, n_cpus=8, contrast = ('Condition','MUT','C'))
stat_res.summary()
res = stat_res.results_df
res['Symbol'] = res.index.map(d) 
res_MUT= res[res.baseMean >= 10]

In [ ]:
fig = plt.figure(figsize = ((2.5),2.5))#20,10
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
res_MUT['log_fdr']=-1*np.log10(res_MUT['padj'])
res_MUT['log_fdr_v2'] = -1*np.log10(fdr(res_MUT['pvalue']))
ax = sns.scatterplot(x='log2FoldChange',y='log_fdr_v2',data=res_MUT,edgecolor=None,alpha=.5,color='#B4B5B4')
plt.title('$5‑HTR7^{V451P}\,vs\;pCDNA$',fontsize=10)
ax.grid()
texts = []
for i in res_MUT[(res_MUT.log_fdr_v2>1.3)&
                (res_MUT.Symbol.isin(pre_res.results['Pathways of neurodegeneration']['matched_genes'].split(';')))].index:
    texts.append(plt.text(res_MUT.loc[i,'log2FoldChange'],res_MUT.loc[i,'log_fdr_v2'],
            '$%s$'%res_MUT.loc[i,'Symbol'],
                          fontsize=7))
adjust_text(texts, autoalign='xy',force_text=(-.2, .1),force_points=(-.1, 0.1),
            only_move={'points':'xy', 'text':'xy'},
            arrowprops=dict(arrowstyle="-", color='k', lw=0.5),zorder=0)
plt.ylabel('$‑log_{10}(FDR)$',fontsize=10)
plt.xlabel('$log_{2}(FC)$',fontsize=10)
#plt.savefig('GPCR_ThT/RNAseq/MUT_vs_Control_vp.svg',bbox_inches='tight',transparent=True)

In [ ]:
stat_res = DeseqStats(dds, n_cpus=8, contrast = ('Condition','WT','MUT'))
stat_res.summary()
res = stat_res.results_df
res['Symbol'] = res.index.map(d) 
res_delta= res[res.baseMean >= 10]

In [ ]:
fig = plt.figure(figsize = ((2.5),2.5))#20,10
sns.set(font_scale=1,style='ticks',rc={'grid.linestyle': '-','grid.linewidth': .5})
res_delta['log_fdr']=-1*np.log10(res_delta['padj'])
res_delta['log_fdr_v2'] = -1*np.log10(fdr(res_delta['pvalue']))
ax = sns.scatterplot(x='log2FoldChange',y='log_fdr_v2',data=res_delta,edgecolor=None,alpha=.5,color='#3A68AE')
plt.title('$5‑HTR7\,vs\,5‑HTR7^{V451P}$',fontsize=10)
ax.grid()
texts = []
for i in res_delta[(res_delta.log_fdr_v2>0)&
                (res_delta.Symbol.isin(pre_res.results['MAPK signaling pathway']['matched_genes'].split(';')))].index:
    texts.append(plt.text(res_delta.loc[i,'log2FoldChange'],res_delta.loc[i,'log_fdr_v2'],
            '$%s$'%res_delta.loc[i,'Symbol'],
                          fontsize=7))
adjust_text(texts, autoalign='xy',force_text=(-.2, .1),force_points=(-.1, 0.1),
            only_move={'points':'xy', 'text':'xy'},
            arrowprops=dict(arrowstyle="-", color='k', lw=0.5),zorder=0)
plt.ylabel('$‑log_{10}(FDR)$',fontsize=10)
plt.xlabel('$log_{2}(FC)$',fontsize=10)
plt.savefig('GPCR_ThT/RNAseq/WT_vs_MUT_vp.svg',bbox_inches='tight',transparent=True)

In [ ]:
import scanpy as sc

In [ ]:
sc.tl.pca(dds)
dds

In [ ]:
dds.layers['normed_counts']

In [ ]:
AF2.loc[9137].AAseq.find('VSVWVLQS')

In [ ]:
l = []
for i in out_df[out_df.NES<-1.7].index:
    for j in out_df[out_df.NES<-1.7].index:
        
        list1 = out_df.loc[i].Lead_genes.split(';')
        list2 = out_df.loc[j].Lead_genes.split(';')
        if i==j:
            l.append({'pathway1':out_df.loc[i,'Term'],'pathway2':out_df.loc[j,'Term'],'jaccard_similarity':
                 1})
        else:
            l.append({'pathway1':out_df.loc[i,'Term'],'pathway2':out_df.loc[j,'Term'],'jaccard_similarity':
                 jaccard_similarity(list1, list2)})

In [ ]:
out_df[out_df.abs_NES>1.7]#.iloc[[12,8]]

In [ ]:
l1 = (pre_res.results['Coronavirus disease']['matched_genes'].split(';'))
l2 = (pre_res.results['Ribosome']['matched_genes'].split(';'))

In [ ]:
jaccard_similarity(l2, l1)